In [1]:
import numpy as np

import archimedes as arc

In [2]:
M1, M2, L1, L2, G_ACC = 1.0, 1.0, 1.0, 1.0, 9.81  # [kg], [m], [m/s^2]


# --------------------------------------------------------------------------
# Generic forward dynamics from any regular Lagrangian
# --------------------------------------------------------------------------
def make_dynamics(lagrangian, kind="SX"):
    @arc.compile(kind=kind)
    def q_ddot(q, q_dot):
        dL_dq = arc.grad(lagrangian, 0)(q, q_dot)
        M = arc.hess(lagrangian, 1)(q, q_dot)
        C = arc.jac(arc.grad(lagrangian, 1), 0)(q, q_dot)  # d2L/dq dq_dot^T
        return np.linalg.solve(M, dL_dq - C @ q_dot)

    return q_ddot


def make_constrained_dynamics(lagrangian, constraint, n, m, kind="SX"):
    """Holonomic c(q) = 0 via the KKT system."""

    @arc.compile(kind=kind)
    def G_qdot(q, q_dot):
        return arc.jac(constraint)(q) @ q_dot

    @arc.compile(kind=kind)
    def q_ddot(q, q_dot):
        dL_dq = arc.grad(lagrangian, 0)(q, q_dot)
        M = arc.hess(lagrangian, 1)(q, q_dot)
        C = arc.jac(arc.grad(lagrangian, 1), 0)(q, q_dot)
        G = arc.jac(constraint)(q)
        Gdot_qdot = arc.jac(G_qdot, 0)(q, q_dot) @ q_dot  # c''(q)[q_dot, q_dot]

        kkt = np.block([[M, G.T], [G, np.zeros((m, m))]])
        rhs = np.hstack([dL_dq - C @ q_dot, -Gdot_qdot])
        return np.linalg.solve(kkt, rhs)[:n]

    return q_ddot


# --------------------------------------------------------------------------
# Test systems
# --------------------------------------------------------------------------
def double_pendulum_L(q, q_dot):
    t1, t2, w1, w2 = q[0], q[1], q_dot[0], q_dot[1]
    T = (
        0.5 * (M1 + M2) * L1**2 * w1**2
        + 0.5 * M2 * L2**2 * w2**2
        + M2 * L1 * L2 * w1 * w2 * np.cos(t1 - t2)
    )
    V = -(M1 + M2) * G_ACC * L1 * np.cos(t1) - M2 * G_ACC * L2 * np.cos(t2)
    return T - V


def cartesian_L(q, q_dot):
    T = 0.5 * M1 * (q_dot[0] ** 2 + q_dot[1] ** 2) + 0.5 * M2 * (
        q_dot[2] ** 2 + q_dot[3] ** 2
    )
    return T - G_ACC * (M1 * q[1] + M2 * q[3])


def cartesian_c(q):
    return np.hstack(
        [
            q[0] ** 2 + q[1] ** 2 - L1**2,
            (q[2] - q[0]) ** 2 + (q[3] - q[1]) ** 2 - L2**2,
        ]
    )


def reference(q, v):
    t1, t2, w1, w2 = q[0], q[1], v[0], v[1]
    d = t1 - t2
    den = 2 * M1 + M2 - M2 * np.cos(2 * d)
    a1 = (
        -G_ACC * (2 * M1 + M2) * np.sin(t1)
        - M2 * G_ACC * np.sin(t1 - 2 * t2)
        - 2 * np.sin(d) * M2 * (w2**2 * L2 + w1**2 * L1 * np.cos(d))
    ) / (L1 * den)
    a2 = (
        2
        * np.sin(d)
        * (
            w1**2 * L1 * (M1 + M2)
            + G_ACC * (M1 + M2) * np.cos(t1)
            + w2**2 * L2 * M2 * np.cos(d)
        )
    ) / (L2 * den)
    return np.array([a1, a2])

In [3]:
q0 = np.array([1.2, -0.4])
v0 = np.array([0.3, 0.7])

f = make_dynamics(double_pendulum_L)
a, a_ref = np.asarray(f(q0, v0)), reference(q0, v0)
print("== Minimal coordinates ==")
print(f"  archimedes {a}")
print(f"  textbook   {a_ref}")
print(f"  max err = {np.abs(a - a_ref).max():.3e}\n")

print("== Constrained: 4 Cartesian coords + 2 rod-length constraints ==")
cf = make_constrained_dynamics(cartesian_L, cartesian_c, n=4, m=2)
t1, t2, w1, w2 = q0[0], q0[1], v0[0], v0[1]
e1, e2 = a_ref
x0 = np.array(
    [
        L1 * np.sin(t1),
        -L1 * np.cos(t1),
        L1 * np.sin(t1) + L2 * np.sin(t2),
        -L1 * np.cos(t1) - L2 * np.cos(t2),
    ]
)
xv0 = np.array(
    [
        L1 * w1 * np.cos(t1),
        L1 * w1 * np.sin(t1),
        L1 * w1 * np.cos(t1) + L2 * w2 * np.cos(t2),
        L1 * w1 * np.sin(t1) + L2 * w2 * np.sin(t2),
    ]
)
ax = L1 * (e1 * np.cos(t1) - w1**2 * np.sin(t1))
ay = L1 * (e1 * np.sin(t1) + w1**2 * np.cos(t1))
expect = np.array(
    [
        ax,
        ay,
        ax + L2 * (e2 * np.cos(t2) - w2**2 * np.sin(t2)),
        ay + L2 * (e2 * np.sin(t2) + w2**2 * np.cos(t2)),
    ]
)
a_c = np.asarray(cf(x0, xv0))
print(f"  KKT solve {a_c}")
print(f"  expected  {expect}")
print(f"  max err = {np.abs(a_c - expect).max():.3e}\n")

print("== arc.odeint (SUNDIALS/CVODES), 20 s ==")


@arc.compile
def rhs(t, x):
    return np.hstack([x[2:], f(x[:2], x[2:])])


xs = arc.odeint(
    rhs,
    t_span=(0.0, 20.0),
    x0=np.hstack([q0, v0]),
    t_eval=np.linspace(0, 20, 201),
    rtol=1e-12,
    atol=1e-14,
)


def energy(s):
    t1, t2, w1, w2 = s
    return (
        0.5 * (M1 + M2) * L1**2 * w1**2
        + 0.5 * M2 * L2**2 * w2**2
        + M2 * L1 * L2 * w1 * w2 * np.cos(t1 - t2)
        - (M1 + M2) * G_ACC * L1 * np.cos(t1)
        - M2 * G_ACC * L2 * np.cos(t2)
    )


E = np.array([energy(xs[:, k]) for k in range(xs.shape[1])])
print(f"  relative |dE/E0| = {abs(E[-1] - E[0]) / abs(E[0]):.3e}\n")

== Minimal coordinates ==
  archimedes [-9.33509123  3.63757536]
  textbook   [-9.33509123  3.63757536]
  max err = 0.000e+00

== Constrained: 4 Cartesian coords + 2 rod-length constraints ==
  KKT solve [-3.46652622 -8.6680577   0.07471755 -9.63327638]
  expected  [-3.46652622 -8.6680577   0.07471755 -9.63327638]
  max err = 3.553e-15

== arc.odeint (SUNDIALS/CVODES), 20 s ==
  relative |dE/E0| = 1.934e-09

